# Bronze — Central Bank of Iceland Ingestion

Fetches daily policy interest rates and monthly CPI from the Central Bank of Iceland XML API.

**Source:** Seðlabanki Íslands XML Time Series API  
**Groups:** GroupID=1 (policy rates), GroupID=3 (CPI)  
**Output:** `bronze.central_bank_raw` (Delta table)  
**Schedule:** Daily  

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

BASE_URL = "https://sedlabanki.is/xmltimeseries/Default.aspx"
START_DATE = "2022-01-01"


def parse_central_bank_xml(xml_text):
    """Parse Seðlabanki XML response into a flat DataFrame."""
    root = ET.fromstring(xml_text)
    group_name = root.find("Name").text
    records = []

    for ts in root.findall("TimeSeries"):
        ts_id = ts.get("ID")
        ts_name = ts.find("Name").text

        for entry in ts.findall(".//Entry"):
            records.append({
                "group_name": group_name,
                "series_id":  ts_id,
                "series_name": ts_name,
                "date":  entry.find("Date").text,
                "value": float(entry.find("Value").text),
            })

    return pd.DataFrame(records)

In [ ]:
# Policy rates (GroupID=1) — daily frequency, standard date format
resp_rates = requests.get(BASE_URL, params={"DagsFra": START_DATE, "GroupID": 1, "Type": "xml"})
resp_rates.raise_for_status()

df_rates = parse_central_bank_xml(resp_rates.text)
df_rates["date"] = pd.to_datetime(df_rates["date"], format="%m/%d/%Y %I:%M:%S %p")
df_rates["ingested_at"] = pd.Timestamp.now()

print(f"Policy rates — {len(df_rates):,} rows | Series: {df_rates['series_name'].nunique()}")

In [ ]:
# CPI / inflation (GroupID=3) — monthly frequency, same date format
resp_cpi = requests.get(BASE_URL, params={"DagsFra": START_DATE, "GroupID": 3, "Type": "xml"})
resp_cpi.raise_for_status()

df_cpi = parse_central_bank_xml(resp_cpi.text)
df_cpi["date"] = pd.to_datetime(df_cpi["date"], format="%m/%d/%Y %I:%M:%S %p")
df_cpi["ingested_at"] = pd.Timestamp.now()

print(f"CPI — {len(df_cpi):,} rows | Series: {df_cpi['series_name'].unique()}")

In [ ]:
df_all = pd.concat([df_rates, df_cpi], ignore_index=True)

spark_df = spark.createDataFrame(df_all)

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.central_bank_raw")

print(f"Saved to bronze.central_bank_raw — {spark_df.count():,} rows")